In [1]:
import torch
import math
from collections import defaultdict
from tokenizers import Tokenizer

In [2]:
best_rl_thinking = '/kaggle/input/datasets/absolute1000/model-after-rl/RL_BEST.pt'
best_rl_classic = '/kaggle/input/datasets/absolute1000/classic-model-after-rl/RL_BEST.pt'
best_sft_thinking = '/kaggle/input/datasets/absolute1000/model-after-sft/SFT_BEST.pt'
best_sft_classic = '/kaggle/input/datasets/absolute1000/classic-model-after-sft/SFT_BEST.pt'

In [4]:
def RoPE(batch):
    B, L, D = batch.shape
    x1 = batch[..., 0::2]
    x2 = batch[..., 1::2]

    pos = torch.arange(L, device=batch.device)

    freqs = 10000 ** (-torch.arange(0, D, 2, device=batch.device) / D)

    angles = pos[:, None] * freqs[None, :]

    cos = torch.cos(angles).unsqueeze(0)
    sin = torch.sin(angles).unsqueeze(0)

    x1_rot = x1 * cos - x2 * sin
    x2_rot = x1 * sin + x2 * cos

    return torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)

class SelfAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, x, mask=None): # x[batch_size, tok_size, emb_size], 
        query, key, value = self.Query(x), self.Key(x), self.Value(x) #[batch, tok_size, head_size]
        query, key = RoPE(query), RoPE(key)
        
        internal_state = query @ key.transpose(-2, -1) #[batch, tok_size, tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1) #[batch, tok_size, tok_size]
        out = weights @ value
        return out # [batch, tok_size, head_size]

class MultiHeadSelfAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([SelfAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, x, mask=None):
        temporary_states = [head(x, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, tok_size, head_size] -> [batch, tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, tok_size, emb_size]

class FeedForward(torch.nn.Module):
    def __init__(self, emb_size, out_size=None):
        super().__init__()
        neurons = int((8 / 3) * emb_size) - (int((8 / 3) * emb_size) % 64)
        if out_size is None: out_size = emb_size
        self.W1 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.SiLU = torch.nn.SiLU()
        self.W2 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.W3 = torch.nn.Linear(neurons, out_size, bias=False)

    def forward(self, x): # [batch, tok_size, emb_size]
        hidden_state = self.SiLU(self.W1(x)) * self.W2(x) # [batch, tok_size, neurons]
        output = self.W3(hidden_state)
        return output # [batch, tok_size, out_size]

class EncoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask): # [batch_size, context_tok_size, emb_size]

        # [batch_size, context_tok_size, emb_size]
        residual_connection1 = self.MultiHeadSelfAttention(self.RMSNorm1(context), mask) + context
        
        # [batch_size, context_tok_size, emb_size]
        residual_connection2 = self.FeedForward(self.RMSNorm2(residual_connection1)) + residual_connection1

        return residual_connection2 # [batch_size, context_tok_size, emb_size]

class Encoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([EncoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask):
        internal_state = context
        for encoder in self.blocks:
            internal_state = encoder(internal_state, mask)
        output = self.RMSNorm(internal_state)
        return output # [batch_size, context_tok_size, emb_size]

class CrossAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        # data_for_query[batch, answer_tok_size, emb_size]
        # data_for_key_value[batch, context_tok_size, emb_size] 
        
        query, key, value = self.Query(data_for_query), self.Key(data_for_key_value), self.Value(data_for_key_value)
        # query[batch, answer_tok_size, head_size]
        # key, value[batch, context_tok_size, head_size]
        
        internal_state = query @ key.transpose(-2, -1) #[batch, answer_tok_size, context_tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1)
        out = weights @ value
        return out #[batch, answer_tok_size, head_size]

class MultiHeadCrossAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([CrossAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        temporary_states = [head(data_for_query, data_for_key_value, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, answer_tok_size, head_size] -> [batch, answer_tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, answer_tok_size, emb_size]

class DecoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.MultiHeadCrossAttention = MultiHeadCrossAttention(emb_size, n_heads)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)
        self.RMSNorm3 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)

    def forward(self, encoder_output, answer, encoder_mask=None):
        
        # [batch, answer_tok_size, emb_size]
        internal_state = self.MultiHeadSelfAttention(self.RMSNorm1(answer))
        residual_connection1 =  internal_state + answer
        internal_state = self.MultiHeadCrossAttention(
            self.RMSNorm2(residual_connection1), encoder_output, encoder_mask) # [batch, answer_tok_size, emb_size]
        residual_connection2 = internal_state + residual_connection1
        internal_state = self.FeedForward(self.RMSNorm3(residual_connection2))
        residual_connection3 = internal_state + residual_connection2 # [batch, answer_tok_size, emb_size]
        return residual_connection3 # [batch, answer_tok_size, emb_size]
        
class Decoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([DecoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)
        self.final = torch.nn.Linear(emb_size, 12288, bias=False)

    def forward(self, answer, encoder_output, mask):
        internal_state = answer
        for decoder in self.blocks:
            internal_state = decoder(encoder_output, internal_state, mask) # [batch_size, answer_size, emb_size]
        internal_state = self.RMSNorm(internal_state)
        internal_state = self.final(internal_state)
        return internal_state # [batch_size, answer_size, dict_size]

class NN_Classic(torch.nn.Module):
    def _init_weights(self, m):
        if isinstance(m, (torch.nn.Linear, torch.nn.Embedding)):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
        
    def __init__(self, 
                 embedding_size, 
                 encoder_num, 
                 heads_encoder,
                 thinking_internal_state_size, 
                 decoder_num, 
                 heads_decoder):
        super().__init__()
        
        self.Embedding = torch.nn.Embedding(12288, embedding_size)
        self.Encoder = Encoder(encoder_num, embedding_size, heads_encoder)
        self.Decoder = Decoder(decoder_num, embedding_size, heads_decoder)
        self.Decoder.final.weight = self.Embedding.weight
        self.apply(self._init_weights)

    def encode(self, context_ids):
        # context_ids [B, Cs]
        PAD_ID = 0
        mask = (context_ids != PAD_ID)[:, None, :] # [B, 1, Cs]
        memory = self.Encoder(self.Embedding(context_ids), mask) # [B, Cs, Emb]
        return memory, mask
    
    def decode(self, answer_ids, memory, mask):
        return self.Decoder(self.Embedding(answer_ids), memory, mask) # [B, La, V]
    
    def forward(self, context_ids, answer_ids):
        memory, mask = self.encode(context_ids)
        return self.decode(answer_ids, memory, mask)

In [5]:
def RoPE(batch):
    B, L, D = batch.shape
    x1 = batch[..., 0::2]
    x2 = batch[..., 1::2]

    pos = torch.arange(L, device=batch.device)

    freqs = 10000 ** (-torch.arange(0, D, 2, device=batch.device) / D)

    angles = pos[:, None] * freqs[None, :]

    cos = torch.cos(angles).unsqueeze(0)
    sin = torch.sin(angles).unsqueeze(0)

    x1_rot = x1 * cos - x2 * sin
    x2_rot = x1 * sin + x2 * cos

    return torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)

class SelfAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, x, mask=None): # x[batch_size, tok_size, emb_size], 
        query, key, value = self.Query(x), self.Key(x), self.Value(x) #[batch, tok_size, head_size]
        query, key = RoPE(query), RoPE(key)
        
        internal_state = query @ key.transpose(-2, -1) #[batch, tok_size, tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1) #[batch, tok_size, tok_size]
        out = weights @ value
        return out # [batch, tok_size, head_size]

class MultiHeadSelfAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([SelfAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, x, mask=None):
        temporary_states = [head(x, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, tok_size, head_size] -> [batch, tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, tok_size, emb_size]

class FeedForward(torch.nn.Module):
    def __init__(self, emb_size, out_size=None):
        super().__init__()
        neurons = int((8 / 3) * emb_size) - (int((8 / 3) * emb_size) % 64)
        if out_size is None: out_size = emb_size
        self.W1 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.SiLU = torch.nn.SiLU()
        self.W2 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.W3 = torch.nn.Linear(neurons, out_size, bias=False)

    def forward(self, x): # [batch, tok_size, emb_size]
        hidden_state = self.SiLU(self.W1(x)) * self.W2(x) # [batch, tok_size, neurons]
        output = self.W3(hidden_state)
        return output # [batch, tok_size, out_size]

class EncoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask): # [batch_size, context_tok_size, emb_size]

        # [batch_size, context_tok_size, emb_size]
        residual_connection1 = self.MultiHeadSelfAttention(self.RMSNorm1(context), mask) + context
        
        # [batch_size, context_tok_size, emb_size]
        residual_connection2 = self.FeedForward(self.RMSNorm2(residual_connection1)) + residual_connection1

        return residual_connection2 # [batch_size, context_tok_size, emb_size]

class Encoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([EncoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask):
        internal_state = context
        for encoder in self.blocks:
            internal_state = encoder(internal_state, mask)
        output = self.RMSNorm(internal_state)
        return output # [batch_size, context_tok_size, emb_size]

class Thinking(torch.nn.Module):
    def __init__(self, emb_size, internal_state_size=128, vocab_size=12288, momentum=0.1):
        super().__init__()
        self.momentum = momentum
        self.to_vocab = torch.nn.Linear(emb_size, vocab_size, bias=False)
        self.alpha = torch.nn.Parameter(torch.ones(vocab_size))
        self.SiLU = torch.nn.SiLU()

        self.to_internal = torch.nn.Linear(vocab_size, internal_state_size, bias=False)
        self.ln = torch.nn.LayerNorm(internal_state_size)
        self.to_emb = torch.nn.Linear(internal_state_size, emb_size, bias=False)

        self.register_buffer("e_mean", torch.zeros(vocab_size))
        self.t_in = torch.nn.Linear(vocab_size, internal_state_size, bias=False)
        self.t_ln = torch.nn.LayerNorm(internal_state_size)
        self.t_out = torch.nn.Linear(internal_state_size, emb_size, bias=False)

    def forward(self, x, mask_for_thinking, context_ids):
        # x [B, Cs, Emb], mask_for_thinking [B, Cs, 1], context_ids [B, Cs]
        x_unit = x / x.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        vocab_unit = self.to_vocab.weight / self.to_vocab.weight.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        energy = x_unit @ vocab_unit.T # [B, Cs, V], from -1 to 1

        own = torch.zeros_like(energy, dtype=torch.bool).scatter_(2, context_ids[..., None], True)
        drop = (~mask_for_thinking) | own # padding + the position's own token
        kept = energy.masked_fill(drop, 0.0) # [B, Cs, V]

        # words channel
        internal_state = kept * self.alpha
        internal_state = self.to_internal(internal_state)
        internal_state = self.ln(internal_state)
        internal_state = self.SiLU(internal_state)
        words = self.to_emb(internal_state) # [B, Cs, Emb]

        # thought channel
        e = kept.sum(dim=1).float() # [B, V]: sum of energies over positions
        if self.training:
            mu = e.mean(dim=0) # mean over texts in the batch
            with torch.no_grad():
                self.e_mean.mul_(1 - self.momentum).add_(self.momentum * mu)
        else:
            mu = self.e_mean # inference: running mean
        e = e - mu # how this text differs from the others
        e = e / e.pow(2).mean(dim=-1, keepdim=True).sqrt().clamp(min=1e-6) # unit scale

        thought = self.t_in(e)
        thought = self.t_ln(thought)
        thought = self.SiLU(thought)
        thought = self.t_out(thought) # [B, Emb]

        thinking_out = words + thought.unsqueeze(1) # thought is added to every position
        return thinking_out

class CrossAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        # data_for_query[batch, answer_tok_size, emb_size]
        # data_for_key_value[batch, context_tok_size, emb_size] 
        
        query, key, value = self.Query(data_for_query), self.Key(data_for_key_value), self.Value(data_for_key_value)
        # query[batch, answer_tok_size, head_size]
        # key, value[batch, context_tok_size, head_size]
        
        internal_state = query @ key.transpose(-2, -1) #[batch, answer_tok_size, context_tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1)
        out = weights @ value
        return out #[batch, answer_tok_size, head_size]

class MultiHeadCrossAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([CrossAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        temporary_states = [head(data_for_query, data_for_key_value, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, answer_tok_size, head_size] -> [batch, answer_tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, answer_tok_size, emb_size]

class DecoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.MultiHeadCrossAttention = MultiHeadCrossAttention(emb_size, n_heads)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)
        self.RMSNorm3 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)

    def forward(self, encoder_output, answer, encoder_mask=None):
        
        # [batch, answer_tok_size, emb_size]
        internal_state = self.MultiHeadSelfAttention(self.RMSNorm1(answer))
        residual_connection1 =  internal_state + answer
        internal_state = self.MultiHeadCrossAttention(
            self.RMSNorm2(residual_connection1), encoder_output, encoder_mask) # [batch, answer_tok_size, emb_size]
        residual_connection2 = internal_state + residual_connection1
        internal_state = self.FeedForward(self.RMSNorm3(residual_connection2))
        residual_connection3 = internal_state + residual_connection2 # [batch, answer_tok_size, emb_size]
        return residual_connection3 # [batch, answer_tok_size, emb_size]
        
class Decoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([DecoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)
        self.final = torch.nn.Linear(emb_size, 12288, bias=False)

    def forward(self, answer, encoder_output, mask):
        internal_state = answer
        for decoder in self.blocks:
            internal_state = decoder(encoder_output, internal_state, mask) # [batch_size, answer_size, emb_size]
        internal_state = self.RMSNorm(internal_state)
        internal_state = self.final(internal_state)
        return internal_state # [batch_size, answer_size, dict_size]

class NN_Thinking(torch.nn.Module):
    def _init_weights(self, m):
        if isinstance(m, (torch.nn.Linear, torch.nn.Embedding)):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
        
    def __init__(self, 
                 embedding_size, 
                 encoder_num, 
                 heads_encoder,
                 thinking_internal_state_size, 
                 decoder_num, 
                 heads_decoder):
        super().__init__()
        
        self.Embedding = torch.nn.Embedding(12288, embedding_size)
        self.Encoder = Encoder(encoder_num, embedding_size, heads_encoder)
        self.Thinking = Thinking(embedding_size, thinking_internal_state_size)
        self.Thinking.to_vocab.weight = self.Embedding.weight
        self.out_norm = torch.nn.RMSNorm(embedding_size)
        self.Decoder = Decoder(decoder_num, embedding_size, heads_decoder)
        self.Decoder.final.weight = self.Embedding.weight
        self.apply(self._init_weights)
        torch.nn.init.zeros_(self.Thinking.t_out.weight)

    def encode(self, context_ids):
        # context_ids [B, Cs]
        PAD_ID = 0
        mask = (context_ids != PAD_ID)[:, None, :] # [B, 1, Cs]
        memory = self.Encoder(self.Embedding(context_ids), mask) # [B, Cs, Emb]
        memory = self.Thinking(memory, mask.transpose(1, 2), context_ids)
        memory = self.out_norm(memory)
        return memory, mask
    
    def decode(self, answer_ids, memory, mask):
        return self.Decoder(self.Embedding(answer_ids), memory, mask) # [B, La, V]
    
    def forward(self, context_ids, answer_ids):
        memory, mask = self.encode(context_ids)
        return self.decode(answer_ids, memory, mask)

In [6]:
TOKENIZER = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/tokenizer.json"
tok = Tokenizer.from_file(TOKENIZER)
USER_ID, BOT_ID = tok.token_to_id("<user>"), tok.token_to_id("<bot>")
PAD_ID, MASK_ID, EOS_ID = tok.token_to_id("<pad>"), tok.token_to_id("<mask>"), tok.token_to_id("<eos>")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def load_model(path, cls):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    m = cls(**ck["MODEL_SETTINGS_CONFIG"]).to(DEVICE)
    m.load_state_dict(ck["model"])
    return m.eval()


MODELS = {
    "SFT thinking": load_model(best_sft_thinking, NN_Thinking),
    "RL thinking":  load_model(best_rl_thinking, NN_Thinking),
    "SFT classic":  load_model(best_sft_classic, NN_Classic),
    "RL classic":   load_model(best_rl_classic, NN_Classic),
}
print("загружено:", ", ".join(MODELS))

загружено: SFT thinking, RL thinking, SFT classic, RL classic


In [7]:
import numpy as np

DATA_DIR = "/kaggle/input/datasets/absolute1000/tokenized-llm-training"
V_TOK = np.memmap(f"{DATA_DIR}/sft_valid_tokens.bin", dtype=np.uint16, mode="r")
_vm = np.fromfile(f"{DATA_DIR}/sft_valid_mask.bin", dtype=np.uint8)
_voff = np.load(f"{DATA_DIR}/sft_valid_offsets.npy")
_d = np.diff(_vm.astype(np.int8), prepend=0, append=0)
V_S, V_E = np.where(_d == 1)[0], np.where(_d == -1)[0]
V_DS = _voff[np.searchsorted(_voff, V_S, side="right") - 1]


def sample_pair(context_len=256, answer_len=64):
    j = np.random.randint(len(V_S))
    ctx = V_TOK[max(V_DS[j], V_S[j] - context_len):V_S[j]].tolist()
    ans = V_TOK[V_S[j]:V_E[j]].tolist()[:answer_len]
    return ctx, ans

In [8]:
@torch.no_grad()
def llada_generate(model, ids, gen_length=64, block_length=8, steps=64, temperature=0.0,
                   remasking="low_confidence"):
    memory, mask = model.encode(torch.tensor([ids], device=DEVICE))
    x = torch.full((1, gen_length), MASK_ID, dtype=torch.long, device=DEVICE)
    num_blocks = gen_length // block_length
    spb = max(1, steps // num_blocks)
    for b in range(num_blocks):
        s, e = b * block_length, (b + 1) * block_length
        ks = [block_length // spb + (1 if i < block_length % spb else 0) for i in range(spb)]
        for k in ks:
            if k == 0:
                continue
            is_mask = x[0] == MASK_ID
            logits = model.decode(x, memory, mask)[0].double()
            logits[:, MASK_ID] = -float("inf")
            if temperature > 0:
                g = (-torch.log(torch.rand_like(logits))) ** temperature
                x0 = (logits.exp() / g).argmax(-1)
            else:
                x0 = logits.argmax(-1)
            if remasking == "low_confidence":
                conf = torch.softmax(logits, -1).gather(-1, x0[:, None]).squeeze(-1)
            else:
                conf = torch.rand(gen_length, device=DEVICE, dtype=torch.double)
            conf[e:] = -float("inf")
            conf[~is_mask] = -float("inf")
            idx = conf.topk(k).indices
            x[0, idx] = x0[idx]
    out = x[0].tolist()
    for j, t in enumerate(out):
        if t in (EOS_ID, PAD_ID):
            out = out[:j]
            break
    return tok.decode(out), out

In [9]:
@torch.no_grad()
def loo_features(model, x, memory, mask, L):
    batch = x.repeat(L, 1)
    idx = torch.arange(L, device=DEVICE)
    batch[idx, idx] = MASK_ID
    logits = model.decode(batch, memory.expand(L, -1, -1), mask.expand(L, -1, -1)).float()[idx, idx]
    logits[:, MASK_ID] = -float("inf")
    lp = torch.log_softmax(logits, -1)
    p = lp.exp()
    cur_lp = lp.gather(-1, x[0, :L, None]).squeeze(-1)
    ent = -torch.where(p > 0, p * lp, torch.zeros_like(p)).sum(-1)
    top2 = lp.topk(2, -1).values
    margin = top2[:, 0] - top2[:, 1]
    posf = idx.float() / L
    return torch.stack([cur_lp, ent, margin, posf], -1)


def train_error_head(model, n_seq=200, iters=500, gen_length=64):
    feats, labels = [], []
    with torch.no_grad():
        for _ in range(n_seq):
            ctx, ans = sample_pair(answer_len=gen_length)
            x0 = torch.full((1, gen_length), PAD_ID, dtype=torch.long, device=DEVICE)
            x0[0, :len(ans)] = torch.tensor(ans, device=DEVICE)
            memory, mask = model.encode(torch.tensor([ctx], device=DEVICE))
            t = float(np.random.uniform(0.2, 0.9))
            noise = torch.rand(gen_length, device=DEVICE) < t
            xt = torch.where(noise, MASK_ID, x0[0])[None]
            logits = model.decode(xt, memory, mask)[0].float()
            logits[:, MASK_ID] = -float("inf")
            x_hat = torch.multinomial(torch.softmax(logits, -1), 1).squeeze(-1)
            y = torch.where(noise, x_hat, x0[0])[None]
            feats.append(loo_features(model, y, memory, mask, gen_length))
            labels.append((y[0] != x0[0]).float())
    X, Y = torch.cat(feats), torch.cat(labels)
    head = torch.nn.Linear(4, 1).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=1e-2)
    for _ in range(iters):
        loss = torch.nn.functional.binary_cross_entropy_with_logits(head(X).squeeze(-1), Y)
        opt.zero_grad(); loss.backward(); opt.step()
    acc = ((head(X).squeeze(-1) > 0).float() == Y).float().mean().item()
    print(f"  доля ошибок в данных {Y.mean().item():.3f}, точность головы {acc:.3f}")
    return head.eval()


ERR_HEADS = {}
for name, m in MODELS.items():
    print(f"голова G-Star для {name}")
    ERR_HEADS[name] = train_error_head(m)


@torch.no_grad()
def star_generate(model, ids, gen_length=64, steps=32, temperature=1.0, star_frac=0.25,
                  guided=False, err_head=None):
    memory, mask = model.encode(torch.tensor([ids], device=DEVICE))
    x = torch.full((1, gen_length), MASK_ID, dtype=torch.long, device=DEVICE)
    for k in range(1, steps + 1):
        m_k = round(gen_length * (1 - k / steps))
        logits = model.decode(x, memory, mask)[0].float()
        logits[:, MASK_ID] = -float("inf")
        x_hat = torch.multinomial(torch.softmax(logits / temperature, -1), 1).squeeze(-1)
        is_mask = x[0] == MASK_ID
        x0 = torch.where(is_mask, x_hat, x[0])
        if k <= steps * (1 - star_frac):
            cand = is_mask.nonzero().squeeze(-1)
            n_reveal = max(0, len(cand) - m_k)
            pick = cand[torch.randperm(len(cand), device=DEVICE)[:n_reveal]]
            x[0, pick] = x0[pick]
        else:
            x = x0[None].clone()
            if m_k > 0:
                if guided:
                    score = err_head(loo_features(model, x, memory, mask, gen_length)).squeeze(-1)
                    remask = score.topk(m_k).indices
                else:
                    remask = torch.randperm(gen_length, device=DEVICE)[:m_k]
                x[0, remask] = MASK_ID
    out = x[0].tolist()
    for j, t in enumerate(out):
        if t in (EOS_ID, PAD_ID):
            out = out[:j]
            break
    return tok.decode(out), out

голова G-Star для SFT thinking
  доля ошибок в данных 0.102, точность головы 0.945
голова G-Star для RL thinking
  доля ошибок в данных 0.121, точность головы 0.927
голова G-Star для SFT classic
  доля ошибок в данных 0.115, точность головы 0.939
голова G-Star для RL classic
  доля ошибок в данных 0.114, точность головы 0.933


In [10]:
@torch.no_grad()
def modern_generate(model, ids, answer_len=64, block_size=8, gamma=0.5, pos_lambda=0.5,
                    temperature=0.0, min_len=3, length_scale=1.0):
    memory, mask = model.encode(torch.tensor([ids], device=DEVICE))
    x = torch.full((1, answer_len), MASK_ID, dtype=torch.long, device=DEVICE)
    probs0 = torch.softmax(model.decode(x, memory, mask)[0].float(), -1)
    L = int(round((1 - probs0[:, EOS_ID] - probs0[:, PAD_ID]).sum().item() * length_scale))
    L = max(min_len, min(L, answer_len - 1))
    x[0, L] = EOS_ID
    x[0, L + 1:] = PAD_ID
    pos = torch.arange(answer_len, device=DEVICE)

    while (x[0, :L] == MASK_ID).any():
        is_mask = x[0] == MASK_ID
        logits = model.decode(x, memory, mask)[0].float()
        logits[:, [MASK_ID, PAD_ID, EOS_ID]] = -float("inf")
        lp = torch.log_softmax(logits, -1)
        p = lp.exp()
        ent = -torch.where(p > 0, p * lp, torch.zeros_like(p)).sum(-1)
        if temperature > 0:
            x0 = torch.multinomial(torch.softmax(logits / temperature, -1), 1).squeeze(-1)
        else:
            x0 = logits.argmax(-1)
        conf_lp = lp.gather(-1, x0[:, None]).squeeze(-1)

        first = int(is_mask.nonzero()[0])
        block_end = min((first // block_size + 1) * block_size, L)
        eligible = is_mask & (pos < block_end)
        score = conf_lp - pos_lambda * (pos - first).float() / block_size
        score = score.masked_fill(~eligible, -float("inf"))

        chosen, cum = [], 0.0
        for i in score.argsort(descending=True).tolist():
            if not eligible[i]:
                break
            if chosen and cum + ent[i].item() > gamma:
                break
            chosen.append(i)
            cum += ent[i].item()
        x[0, chosen] = x0[chosen]

    out = x[0, :L].tolist()
    return tok.decode(out), out

In [13]:
def trim_history(history, context_len=256):
    if len(history) <= context_len:
        return history
    for i, t in enumerate(history):
        if t == USER_ID and len(history) - i <= context_len:
            return history[i:]
    return history[-context_len:]

In [11]:
def chat_sampler(sampler="modern", mode="dialog", context_len=256, seed=0, **kw):
    fn = {"llada": llada_generate, "star": star_generate,
          "gstar": star_generate, "modern": modern_generate}[sampler]
    hist = {name: [] for name in MODELS}
    print(f"сэмплер: {sampler}, режим: {mode}, окно {context_len}. Выход — Enough")
    while (a := input()) != "Enough":
        print(f"\nUSER: {a}")
        for name, m in MODELS.items():
            if mode == "single":
                hist[name] = []
            hist[name] += [USER_ID] + tok.encode(a).ids + [BOT_ID]
            window = trim_history(hist[name], context_len)
            extra = {}
            if sampler == "star":
                extra = {"guided": False}
            elif sampler == "gstar":
                extra = {"guided": True, "err_head": ERR_HEADS[name]}
            torch.manual_seed(seed)
            reply, reply_ids = fn(m, window, **extra, **kw)
            hist[name] += reply_ids + [EOS_ID]
            print(f"  {name:>13}: {reply}")

In [14]:
chat_sampler("modern", mode="dialog")

сэмплер: modern, режим: dialog, окно 256. Выход — Enough


 Hello.



USER: Hello.
   SFT thinking: Hi, Hello. How are you, please?
    RL thinking: Hi,Hello,Hello, Hello.
    SFT classic: Hi, I'm a doctor.
     RL classic: Hello.


 I am going to play football after breakfast.



USER: I am going to play football after breakfast.
   SFT thinking: I am good. I am good. Do you play football?
    RL thinking: I am a good man, I am a lot of work for you.
    SFT classic: I'm a doctor at college.
     RL classic: Hello.


 Do you want to go to play football with me?



USER: Do you want to go to play football with me?
   SFT thinking: I like to play football. I like to play soccer.
    RL thinking: I'm not sure I do not have a team so I like it.
    SFT classic: Yes I do not play football.
     RL classic: Hello.


 Ok. See you on the field near the shop. Goodbye.



USER: Ok. See you on the field near the shop. Goodbye.
   SFT thinking: I like to get a job. I like it.
    RL thinking: I am a singer. I am a little singer. How are you?
    SFT classic: I'm not in the shop.
     RL classic: Hello.


KeyboardInterrupt: Interrupted by user

In [15]:
chat_sampler("gstar", mode="dialog")

сэмплер: gstar, режим: dialog, окно 256. Выход — Enough


 Hello.



USER: Hello.
   SFT thinking: Hello OHello...Hello, HHello? Where enjoy you?
    RL thinking: Hi, Mary, you are you, you?
    SFT classic: Hi just doing great. What is the call? I'm here at the gate.
     RL classic: Hi, thanks. I am a prince.


 I am going to play football after breakfast.



USER: I am going to play football after breakfast.
   SFT thinking: Its good. Do you play my favorite singer? Football is awesome.
    RL thinking: That's cool I'm going to be a kid who are'm going to go.
    SFT classic: I'm sorry, I've won as a trailer Eat, I am currently active.
     RL classic: Sorry. Do you play that? or any kind of sports.


 Do you want to go to play football with me?



USER: Do you want to go to play football with me?
   SFT thinking: Yes I do. Do you have hobbies too? I do enjoy work.
    RL thinking: I'm not, but I am pretty sweet. I am blind, yes.
    SFT classic: I'm sorry. Do you want to play football? You? Which lob you for both?
     RL classic: Yes, I don not want to experiment at a fockey degree.


 Ok. See you on the field near the shop. Goodbye.



USER: Ok. See you on the field near the shop. Goodbye.
   SFT thinking: I am not. Do you have a hober? I am very fine.
    RL thinking: I am ok, I am living in my high school tonight. You?
    SFT classic: I'm sorry. Do you have the car or caring? Where enjoy you for both?
     RL classic: Yes I do, I am only in Chicago and I overham spring.


KeyboardInterrupt: Interrupted by user

In [17]:
chat_sampler("star", mode="dialog")

сэмплер: star, режим: dialog, окно 256. Выход — Enough


 Hello.



USER: Hello.
   SFT thinking: Hello Copampello.Hello. I have three dogs, to get back
    RL thinking: Hi K?Good, how hasthing K?
    SFT classic: 'all love it. Where is your Mark? Me, now, owning back
     RL classic: Hello? Yello. Hello.


 I am going to play football after breakfast.



USER: I am going to play football after breakfast.
   SFT thinking: Coolice. do you enjoy h good roll impressive? Haster this comes back
    RL thinking: Coolctares are fun to play the roll. What coaster play?
    SFT classic: Congctares, I enjoy cool series. Are you there, owning back
     RL classic: Coolice. Do you listen to the sing impressive?


 Do you want to go to play football with me?



USER: Do you want to go to play football with me?
   SFT thinking: No like no. Do you think about good roll impressive? Haster this comes back
    RL thinking: I volunteer well. I love to play the rollunt ball coaster play?
    SFT classic: I don well. Do you not play good sing, love music, I get back well.
     RL classic: I don't like to do not since good sing like teasaster.


 Ok. See you on the field near the shop. Goodbye.



USER: Ok. See you on the field near the shop. Goodbye.
   SFT thinking: They are worked. Do you not have the roll impressive? Haster to get back
    RL thinking: Good at worked. do you paint h good roll impressive coasters, everyday?
    SFT classic: Thank you. Do you not play good sing, love music, enjoy going back well.
     RL classic: I don't like to do not since good sing like teasaster.


KeyboardInterrupt: Interrupted by user

In [19]:
chat_sampler("llada", mode="dialog")

сэмплер: llada, режим: dialog, окно 256. Выход — Enough


 Hello.



USER: Hello.
   SFT thinking: 
    RL thinking: Hi,
    SFT classic: Hello?
     RL classic: Hello, thanks.


 I am going to play football after breakfast.



USER: I am going to play football after breakfast.
   SFT thinking: Do you play?
    RL thinking: What kind do you do?
    SFT classic: Do you play?
     RL classic: I love football.


 Do you want to go to play football with me?



USER: Do you want to go to play football with me?
   SFT thinking: 
    RL thinking: I am good
    SFT classic:  You?
     RL classic: I love.


 Ok. See you on the field near the shop. Goodbye.



USER: Ok. See you on the field near the shop. Goodbye.
   SFT thinking: 
    RL thinking: I am good.
    SFT classic: ?
     RL classic: I love football.


KeyboardInterrupt: Interrupted by user

In [20]:
def elite_ngram_tables(seq, n):
    nxt, prv = defaultdict(set), defaultdict(set)
    for j in range(len(seq) - n + 1):
        g = seq[j:j + n]
        if MASK_ID in g:
            continue
        nxt[tuple(g[:-1])].add(g[-1])
        prv[tuple(g[1:])].add(g[0])
    return nxt, prv


def elite_ngram_banned(a, i, L, nxt, prv, n):
    m, banned = n - 1, set()
    if i >= m and MASK_ID not in a[i - m:i]:
        banned |= nxt.get(tuple(a[i - m:i]), set())
    if i + m < L and MASK_ID not in a[i + 1:i + 1 + m]:
        banned |= prv.get(tuple(a[i + 1:i + 1 + m]), set())
    return banned


@torch.no_grad()
def elite_fill(model, ans, memory, mask, region, ctx_ids, L, temperature, top_k, block_size, rep_penalty,
               ngram_ans, ngram_ctx, ngram_penalty):
    pos = torch.arange(ans.shape[1], device=ans.device)
    ctx_tables = elite_ngram_tables(ctx_ids, ngram_ctx) if ngram_ctx > 1 else None
    while True:
        is_mask = ans[0] == MASK_ID
        todo = is_mask & region
        if not todo.any():
            break
        logits = model.decode(ans, memory, mask)[0].float()
        logits[:, [MASK_ID, PAD_ID, EOS_ID]] = -float("inf")
        used = ans[0][~is_mask].unique()
        used = used[(used != EOS_ID) & (used != PAD_ID)]
        if rep_penalty > 1.0 and len(used):
            lu = logits[:, used]
            logits[:, used] = torch.where(lu > 0, lu / rep_penalty, lu * rep_penalty)
        first = int(todo.nonzero()[0])
        eligible = todo & (pos < (first // block_size + 1) * block_size)
        if ngram_penalty > 0:
            a = ans[0].tolist()
            ans_tables = elite_ngram_tables(a[:L], ngram_ans) if ngram_ans > 1 else None
            for i in eligible.nonzero().squeeze(-1).tolist():
                banned = set()
                if ans_tables:
                    banned |= elite_ngram_banned(a, i, L, *ans_tables, ngram_ans)
                if ctx_tables:
                    banned |= elite_ngram_banned(a, i, L, *ctx_tables, ngram_ctx)
                if banned:
                    logits[i, list(banned)] -= ngram_penalty
        logits = logits / temperature
        kth = logits.topk(top_k, dim=-1).values[:, -1:]
        probs = torch.softmax(logits.masked_fill(logits < kth, -float("inf")), -1)
        pred = torch.multinomial(probs, 1).squeeze(-1)
        conf = probs.gather(-1, pred[:, None]).squeeze(-1).masked_fill(~eligible, -1.0)
        i = int(conf.argmax())
        ans[0, i] = pred[i]


@torch.no_grad()
def elite_loo_probs(model, ans, memory, mask, end):
    batch = ans.repeat(end, 1)
    idx = torch.arange(end, device=ans.device)
    batch[idx, idx] = MASK_ID
    logits = model.decode(batch, memory.expand(end, -1, -1), mask.expand(end, -1, -1)).float()[idx, idx]
    logits[:, [MASK_ID, PAD_ID, EOS_ID]] = -float("inf")
    return torch.softmax(logits, -1)


@torch.no_grad()
def elite_gstar(model, ans, memory, mask, start, end, rounds, margin, protected):
    for _ in range(rounds):
        probs = elite_loo_probs(model, ans, memory, mask, end)
        cur = ans[0, :end]
        p_cur = probs.gather(-1, cur[:, None]).squeeze(-1)
        p_best, best_tok = probs.max(-1)
        bad = (p_cur < margin * p_best) & (best_tok != cur)
        bad[:start] = False
        bad &= ~torch.isin(cur, protected)
        if not bad.any():
            break
        j = int(torch.where(bad, p_best - p_cur, torch.full_like(p_cur, -1.0)).argmax())
        ans[0, j] = best_tok[j]


@torch.no_grad()
def elite_generate(model, ids, answer_len=64, temperature=0.6, top_k=50, block_size=8, rep_penalty=1.05,
                   min_len=3, length_scale=1.0, n_candidates=4, block_rounds=2, global_rounds=2,
                   margin=0.5, ngram_ans=3, ngram_ctx=4, ngram_penalty=5.0):
    ctx = torch.tensor([ids], device=DEVICE)
    memory, mask = model.encode(ctx)
    base = torch.full((1, answer_len), MASK_ID, dtype=torch.long, device=DEVICE)
    probs0 = torch.softmax(model.decode(base, memory, mask)[0].float(), -1)
    L_mean = (1 - probs0[:, EOS_ID] - probs0[:, PAD_ID]).sum().item() * length_scale
    pos = torch.arange(answer_len, device=DEVICE)

    u = len(ids) - 1 - ids[::-1].index(USER_ID) if USER_ID in ids else -1
    protected = torch.tensor(ids[u + 1:-1] if u >= 0 else [], dtype=torch.long, device=DEVICE)

    scales = torch.linspace(0.8, 1.2, n_candidates).tolist() if n_candidates > 1 else [1.0]
    best_ids, best_score = [], -float("inf")
    for s in scales:
        L = max(min_len, min(int(round(L_mean * s)), answer_len - 1))
        ans = base.clone()
        ans[0, L] = EOS_ID
        ans[0, L + 1:] = PAD_ID
        fill_args = dict(ctx_ids=ids, L=L, temperature=temperature, top_k=top_k, block_size=block_size,
                         rep_penalty=rep_penalty, ngram_ans=ngram_ans, ngram_ctx=ngram_ctx,
                         ngram_penalty=ngram_penalty)
        for b_start in range(0, L, block_size):
            b_end = min(b_start + block_size, L)
            elite_fill(model, ans, memory, mask, region=pos < b_end, **fill_args)
            elite_gstar(model, ans, memory, mask, b_start, b_end, block_rounds, margin, protected)
        elite_gstar(model, ans, memory, mask, 0, L, global_rounds, margin, protected)

        p = elite_loo_probs(model, ans, memory, mask, L).gather(-1, ans[0, :L, None]).squeeze(-1)
        score = p.clamp_min(1e-9).log().mean().item()
        if score > best_score:
            best_score, best_ids = score, ans[0, :L].tolist()

    return tok.decode(best_ids), best_ids

In [24]:
SAMPLER_FN = {"llada": llada_generate, "star": star_generate, "gstar": star_generate,
              "modern": modern_generate, "elite": elite_generate}


def sampler_extra(sampler, model_name):
    if sampler == "star":
        return {"guided": False}
    if sampler == "gstar":
        return {"guided": True, "err_head": ERR_HEADS[model_name]}
    return {}


def chat_sampler(sampler="elite", mode="dialog", context_len=256, seed=0, **kw):
    fn = SAMPLER_FN[sampler]
    hist = {name: [] for name in MODELS}
    print(f"сэмплер: {sampler}, режим: {mode}, окно {context_len}. Выход — Enough")
    while (a := input()) != "Enough":
        print(f"\nUSER: {a}")
        for name, m in MODELS.items():
            if mode == "single":
                hist[name] = []
            hist[name] += [USER_ID] + tok.encode(a).ids + [BOT_ID]
            window = trim_history(hist[name], context_len)
            torch.manual_seed(seed)
            reply, reply_ids = SAMPLER_FN[sampler](m, window, **sampler_extra(sampler, name), **kw)
            hist[name] += reply_ids + [EOS_ID]
            print(f"  {name:>13}: {reply}")

In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
import pandas as pd

j_tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
j_model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").to(DEVICE).eval()


@torch.no_grad()
def judge_logprob(prompt, reply):
    ctx_ids = j_tok(f"User: {prompt}\nBot:").input_ids
    rep_ids = j_tok(" " + reply.strip()).input_ids
    ids = torch.tensor([ctx_ids + rep_ids], device=DEVICE)
    logits = j_model(ids).logits[0, :-1].float()
    lp = torch.log_softmax(logits, -1).gather(-1, ids[0, 1:, None]).squeeze(-1)
    return lp[len(ctx_ids) - 1:].mean().item()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [27]:
EVAL_PROMPTS = [
    "Hello! How are you today?",
    "I just got a new puppy!",
    "I lost my job yesterday.",
    "What do you like to do on weekends?",
    "I'm so tired after work.",
    "Do you have any pets?",
    "What is your favorite food?",
    "I failed my driving test again.",
    "My sister is getting married next month.",
    "The weather is terrible today, it's raining all day.",
    "Can you recommend a good movie?",
    "Goodbye, see you tomorrow!",
]

GRID = {
    "llada":  [dict(temperature=t, block_length=b) for t in (0.0, 0.5, 0.8) for b in (8, 16)],
    "star":   [dict(temperature=t, star_frac=f) for t in (0.5, 0.8, 1.0) for f in (0.25, 0.5)],
    "gstar":  [dict(temperature=t, star_frac=f) for t in (0.5, 0.8, 1.0) for f in (0.25, 0.5)],
    "modern": [dict(temperature=t, gamma=g) for t in (0.0, 0.5, 0.7) for g in (0.2, 0.5, 1.0)],
    "elite":  [dict(temperature=t, margin=mg, n_candidates=n)
               for t in (0.5, 0.7) for mg in (0.3, 0.5) for n in (2, 4)],
}


def bigrams(words):
    return list(zip(words, words[1:]))


def evaluate_replies(replies, prompts=EVAL_PROMPTS):
    flu, rep, copy, n_words, all_bi = [], [], [], [], []
    empty = short = 0
    for p, r in zip(prompts, replies):
        w = r.lower().split()
        if not w:
            empty += 1
            continue
        if len(w) < 3:
            short += 1
        flu.append(judge_logprob(p, r))
        bi = bigrams(w)
        all_bi += bi
        rep.append((len(bi) - len(set(bi))) / len(bi) if bi else 0.0)
        pb = set(bigrams(p.lower().split()))
        copy.append(sum(b in pb for b in bi) / len(bi) if bi else 0.0)
        n_words.append(len(w))
    n = len(replies)
    m = {
        "fluency":   float(np.mean(flu)) if flu else -10.0,
        "distinct2": len(set(all_bi)) / len(all_bi) if all_bi else 0.0,
        "empty":     empty / n,
        "short":     short / n,
        "repeat":    float(np.mean(rep)) if rep else 0.0,
        "copy":      float(np.mean(copy)) if copy else 0.0,
        "words":     float(np.mean(n_words)) if n_words else 0.0,
    }
    m["score"] = (m["fluency"] + 2 * m["distinct2"] - 3 * m["empty"] - 1 * m["short"]
                  - 2 * m["repeat"] - 1 * m["copy"])
    return m


def run_config(sampler, params, model_name, model):
    replies = []
    for i, p in enumerate(EVAL_PROMPTS):
        ids = [USER_ID] + tok.encode(p).ids + [BOT_ID]
        torch.manual_seed(i)
        r, _ = SAMPLER_FN[sampler](model, ids, **sampler_extra(sampler, model_name), **params)
        replies.append(r)
    return replies

In [28]:
results = []
total = sum(len(v) for v in GRID.values()) * len(MODELS)
done = 0
t_start = time.time()
for sampler, configs in GRID.items():
    for params in configs:
        for name, m in MODELS.items():
            replies = run_config(sampler, params, name, m)
            met = evaluate_replies(replies)
            results.append({"sampler": sampler, "params": str(params), "model": name, **met, "replies": replies})
            done += 1
            print(f"[{done}/{total}] {time.time() - t_start:6.0f}s  {sampler:6} {str(params):55} {name:13} "
                  f"score {met['score']:+.2f}", flush=True)

df = pd.DataFrame(results)
df.to_pickle("/kaggle/working/sampler_grid.pkl")

[1/140]     11s  llada  {'temperature': 0.0, 'block_length': 8}                 SFT thinking  score -2.46
[2/140]     22s  llada  {'temperature': 0.0, 'block_length': 8}                 RL thinking   score -1.88
[3/140]     33s  llada  {'temperature': 0.0, 'block_length': 8}                 SFT classic   score -2.07
[4/140]     44s  llada  {'temperature': 0.0, 'block_length': 8}                 RL classic    score -2.90
[5/140]     55s  llada  {'temperature': 0.0, 'block_length': 16}                SFT thinking  score -2.24
[6/140]     66s  llada  {'temperature': 0.0, 'block_length': 16}                RL thinking   score -1.21
[7/140]     77s  llada  {'temperature': 0.0, 'block_length': 16}                SFT classic   score -1.45
[8/140]     88s  llada  {'temperature': 0.0, 'block_length': 16}                RL classic    score -1.80
[9/140]     99s  llada  {'temperature': 0.5, 'block_length': 8}                 SFT thinking  score -2.32
[10/140]    110s  llada  {'temperature': 0.5, 

In [29]:
cols = ["sampler", "params", "score", "fluency", "distinct2", "empty", "short", "repeat", "copy", "words"]

print("=== лучшие наборы в среднем по 4 моделям ===")
overall = (df.groupby(["sampler", "params"])[cols[2:]].mean()
             .sort_values("score", ascending=False).reset_index())
print(overall.head(10).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\n=== лучший набор у каждого сэмплера ===")
best_per_sampler = overall.loc[overall.groupby("sampler")["score"].idxmax()].sort_values("score", ascending=False)
print(best_per_sampler.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

for name in MODELS:
    print(f"\n=== {name}: топ-5 ===")
    top = df[df.model == name].sort_values("score", ascending=False).head(5)
    print(top[cols].to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\n=== ответы лучшего набора для каждой модели ===")
for name in MODELS:
    row = df[df.model == name].sort_values("score", ascending=False).iloc[0]
    print(f"\n--- {name} | {row.sampler} {row.params} | score {row.score:+.3f} ---")
    for p, r in zip(EVAL_PROMPTS, row.replies):
        print(f"  USER: {p}\n   BOT: {r}")

=== лучшие наборы в среднем по 4 моделям ===
sampler                                                 params  score  fluency  distinct2  empty  short  repeat  copy  words
  elite {'temperature': 0.5, 'margin': 0.5, 'n_candidates': 2} -1.558   -2.885      0.879  0.000  0.375   0.020 0.017  7.125
  elite {'temperature': 0.5, 'margin': 0.5, 'n_candidates': 4} -1.558   -2.795      0.840  0.000  0.354   0.025 0.040  7.521
  elite {'temperature': 0.5, 'margin': 0.3, 'n_candidates': 2} -1.574   -2.906      0.881  0.000  0.375   0.020 0.017  6.979
  elite {'temperature': 0.7, 'margin': 0.5, 'n_candidates': 2} -1.584   -2.889      0.851  0.000  0.354   0.010 0.021  7.792
  elite {'temperature': 0.7, 'margin': 0.3, 'n_candidates': 4} -1.591   -2.881      0.834  0.000  0.333   0.011 0.022  7.292
  elite {'temperature': 0.7, 'margin': 0.5, 'n_candidates': 4} -1.610   -2.890      0.830  0.000  0.333   0.016 0.014  7.625
  elite {'temperature': 0.7, 'margin': 0.3, 'n_candidates': 2} -1.610   -2.927  